In [1]:
import tensorflow as tf
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [2]:
import tensorflow as tf
import GPUtil

# Elencare le GPU disponibili
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    # Ottenere informazioni dettagliate sulla prima GPU
    gpu_info = tf.test.gpu_device_name()
    
    if gpu_info:
        print(f"Stai utilizzando la GPU: {gpu_info}")

        # Ottenere informazioni aggiuntive sulla GPU utilizzando GPUtil
        gpu_list = GPUtil.getGPUs()
        if gpu_list:
            first_gpu = gpu_list[0]
            print(f"Nome del dispositivo GPU: {first_gpu.name}")
            print(f"Memoria totale della GPU: {first_gpu.memoryTotal} MB")
        else:
            print("Nessuna informazione dettagliata sulla GPU disponibile.")
    else:
        print("Nessuna informazione sulla GPU disponibile.")
else:
    print("Nessuna GPU disponibile.")

Stai utilizzando la GPU: /device:GPU:0
Nome del dispositivo GPU: NVIDIA GeForce GTX 1660 Ti
Memoria totale della GPU: 6144.0 MB


In [3]:
import os
import pandas as pd
from PIL import Image


# percorso alla cartella delle immagini
image_folder = "ProjectBackUp/Images/dataReal"
# percorso ai file CSV con i valori di clorofilla e ammoniaca
chlorophyll_file = "ProjectBackUp/CSV/Chlorophyll.csv"
ammonia_file = "ProjectBackUp/CSV/Ammonia.csv"

# Inizializza le liste per le informazioni delle immagini
data = []


# Legge i valori target da file CSV
chlorophyll_data = pd.read_csv(chlorophyll_file, header=None, names=["Chlorophyll"])
ammonia_data = pd.read_csv(ammonia_file, header=None, names=["Ammonia"])

In [4]:
import os
import pandas as pd

# Sostituisci con il percorso dell'output_folder
output_folder = 'ProjectBackUp/Images/dataReal'
chlorophyll_data = pd.read_csv(chlorophyll_file, header=None, names=["Chlorophyll"])
ammonia_data = pd.read_csv(ammonia_file, header=None, names=["Ammonia"])

# Elenco dei file nell'output_folder
image_files = os.listdir(output_folder)


data = []

for image_file in image_files:
    # Costruisci il percorso completo all'immagine
    image_path = os.path.join(output_folder, image_file)

    # Estrai il valore di ammoniaca corrispondente
    ammonium_value = ammonia_data.iloc[image_files.index(image_file)]["Ammonia"]

    # Estrai il valore di clorofilla corrispondente
    chlorophyll_value = chlorophyll_data.iloc[image_files.index(image_file)]["Chlorophyll"]

    # Aggiungi le informazioni all'elenco dei dati
    data.append([image_path, ammonium_value, chlorophyll_value])

# Creazione del dataframe
df = pd.DataFrame(data, columns=["Image_Path", "Ammonia", "Chlorophyll"])


df

,Image_Path,Ammonia,Chlorophyll
0,ProjectBackUp/Images/dataReal\2019_12_05_5_Tre...,12.28,45.52
1,ProjectBackUp/Images/dataReal\2019_12_05_5_Tre...,10.23,43.08
2,ProjectBackUp/Images/dataReal\2019_12_05_5_Tre...,13.14,44.52
3,ProjectBackUp/Images/dataReal\2019_12_05_5_Tre...,12.28,45.52
4,ProjectBackUp/Images/dataReal\2019_12_05_5_Tre...,10.23,43.08
...,...,...,...
1186,ProjectBackUp/Images/dataReal\2021 05 06 S__S_...,57.99,31.49
1187,ProjectBackUp/Images/dataReal\2021 05 06 S__S_...,57.99,31.49
1188,ProjectBackUp/Images/dataReal\2021 05 06 S__S_...,84.22,32.18
1189,ProjectBackUp/Images/dataReal\2021 05 06 S__S_...,84.22,32.18


## Here I have normalized the Ammonia values by dividing each value by the max ammonia value in the dataset which is 284.25. You can do the same for Chlorophyll. Normalized values in a certain range are better

In [5]:
# Creazione del DataFrame senza la colonna "Chlorophyll"
df_ammonia = df.drop("Chlorophyll", axis=1)

# Creazione del DataFrame senza la colonna "Ammonia"
df_cloro = df.drop("Ammonia", axis=1)

In [6]:
df_ammonia['Ammonia'] = df_ammonia['Ammonia'] / 284.25

## Since now the values are in [0,1] range so I am using Sigmoid activation in the final layer alongwith Binary_CrossEntropy loss since Sigmoid produces values in the range of [0,1] and after the training is completed and we have gotten our predicted values, we can multiply them with the max value 284.25 to inversely transform and compare

In [7]:
from sklearn.model_selection import KFold
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Creazione del generatore di immagini per l'intero dataset
full_data_generator = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
)

# Creazione del generatore di immagini per l'intero dataset
# full_generator = full_data_generator.flow_from_dataframe(
#     dataframe=df_ammonia,  # Utilizza l'intero DataFrame
#     x_col="Image_Path",
#     y_col=["Ammonia"],
#     batch_size=8,
#     class_mode="raw",
#     target_size=(768, 1024)
# )


# Definizione dei parametri K-fold per Ammonia
n_splits_ammonia = 10
kf_ammonia = KFold(n_splits=n_splits_ammonia, shuffle=True, random_state=42)

# Inizializzazione delle liste per le metriche per Ammonia
mae_scores_ammonia = []
rmse_scores_ammonia = []
mse_scores_ammonia = []
rse_scores_ammonia = []

# K-fold Cross Validation per Ammonia
for fold_num_ammonia, (train_index_ammonia, val_index_ammonia) in enumerate(kf_ammonia.split(df_ammonia), 1):
    train_df_ammonia, val_df_ammonia = df_ammonia.iloc[train_index_ammonia], df_ammonia.iloc[val_index_ammonia]

    # Creazione del generatore di immagini per il set di addestramento per Ammonia
    train_generator_ammonia = full_data_generator.flow_from_dataframe(
        dataframe=train_df_ammonia,
        x_col="Image_Path",
        y_col=["Ammonia"],
        batch_size=4,
        class_mode="raw",
        target_size=(768, 1024)
    )

    # Creazione del generatore di immagini per il set di convalida per Ammonia
    val_generator_ammonia = full_data_generator.flow_from_dataframe(
        dataframe=val_df_ammonia,
        x_col="Image_Path",
        y_col=["Ammonia"],
        batch_size=4,
        class_mode="raw",
        target_size=(768, 1024)
    )

    # Caricamento del modello VGG16 preaddestrato senza i layer fully connected
    vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(768, 1024, 3))
    for layer in vgg_base.layers:
        layer.trainable = False
    # Aggiunta dei layer fully connected
    model_ammonia = models.Sequential()
    model_ammonia.add(vgg_base)
    model_ammonia.add(layers.Flatten())
    model_ammonia.add(layers.Dense(64, activation='relu'))
    model_ammonia.add(layers.Dense(1, activation='sigmoid'))

    # Compilazione del modello
    model_ammonia.compile(optimizer='adam', loss='binary_crossentropy', metrics=['mae'])

    # Early Stopping
    early_stopping = EarlyStopping(monitor='val_loss', patience=5, verbose=1)

    # Addestramento del modello per Ammonia
    model_ammonia.fit(train_generator_ammonia, epochs=20, validation_data=val_generator_ammonia, batch_size = 4, callbacks=[early_stopping], verbose=1)

    # Salva il modello per la predizione dell'ammoniaca
    model_filename_ammonia = f"model_ammonia_fold_{fold_num_ammonia}.h5"
    model_ammonia.save(model_filename_ammonia)

    # Valutazione del modello su dati di validazione
    true_values = val_df_ammonia["Ammonia"].values * 284.25
    predicted_values = model_ammonia.predict(val_generator_ammonia) * 284.25
    mae_fold_ammonia = mean_absolute_error(true_values, predicted_values)
    rmse_fold_ammonia = mean_squared_error(true_values, predicted_values, squared=False)
    mse_fold_ammonia = mean_squared_error(true_values, predicted_values)
    rse_fold_ammonia = 1 - r2_score(true_values, predicted_values)

    # Aggiunta delle metriche alla lista
    mae_scores_ammonia.append(mae_fold_ammonia)
    rmse_scores_ammonia.append(rmse_fold_ammonia)
    mse_scores_ammonia.append(mse_fold_ammonia)
    rse_scores_ammonia.append(rse_fold_ammonia)

    # Stampa delle metriche per la fold corrente
    print(f"Fold {fold_num_ammonia}:")
    print(f"MAE: {mae_fold_ammonia}, RMSE: {rmse_fold_ammonia}, MSE: {mse_fold_ammonia}, RSE: {rse_fold_ammonia}")
    
    break

# Calcolo delle medie delle metriche su tutte le fold
mean_mae = np.mean(mae_scores_ammonia)
mean_rmse = np.mean(rmse_scores_ammonia)
mean_mse = np.mean(mse_scores_ammonia)
mean_rse = np.mean(rse_scores_ammonia)

# Stampa delle medie delle metriche su tutte le fold
print("\nMean across all folds:")
print(f"MAE: {mean_mae}, RMSE: {mean_rmse}, MSE: {mean_mse}, RSE: {mean_rse}")


Found 1071 validated image filenames.
Found 120 validated image filenames.
Epoch 1/20
268/268 [==============================] - 117s 392ms/step - loss: 4.7537 - mae: 0.2011 - val_loss: 3.0061 - val_mae: 0.1114
Epoch 2/20
268/268 [==============================] - 102s 380ms/step - loss: 2.3840 - mae: 0.1410 - val_loss: 2.2135 - val_mae: 0.1074
Epoch 3/20
268/268 [==============================] - 102s 380ms/step - loss: 2.5505 - mae: 0.1476 - val_loss: 0.8248 - val_mae: 0.1075
Epoch 4/20
268/268 [==============================] - 102s 381ms/step - loss: 1.8638 - mae: 0.1398 - val_loss: 1.1123 - val_mae: 0.1008
Epoch 5/20
268/268 [==============================] - 102s 381ms/step - loss: 1.2429 - mae: 0.1306 - val_loss: 0.8667 - val_mae: 0.1039
Epoch 6/20
268/268 [==============================] - 103s 383ms/step - loss: 0.7715 - mae: 0.1117 - val_loss: 0.3869 - val_mae: 0.0938
Epoch 7/20
268/268 [==============================] - 103s 385ms/step - loss: 0.5281 - mae: 0.1000 - val_loss

In [11]:
for i in range(100):
    print(true_values[i], predicted_values[i])

10.95 [7.806531]
12.46 [18.525415]
13.49 [15.743601]
28.94 [18.93731]
19.45 [9.595316]
20.77 [9.741907]
20.02 [4.9467545]
10.22 [2.084501]
99.12 [8.0941105]
43.74 [6.935139]
5.73 [2.360563]
6.06 [10.1066885]
5.31 [7.692211]
5.22 [15.656682]
2.75 [8.711055]
2.9300000000000006 [3.4196353]
1.34 [37.812057]
0.94 [7.7404532]
0.24 [5.5794396]
4.1 [6.783564]
5.59 [24.549383]
6.300000000000001 [5.7462454]
5.43 [2.0453753]
4.1 [45.44628]
3.23 [3.337327]
2.4 [2.2746308]
1.62 [2.2607102]
1.64 [2.913565]
2.97 [31.04726]
6.0 [6.6947937]
1.56 [2.6927273]
4.63 [5.563023]
19.4 [1.9647982]
162.48 [4.3840623]
7.57 [4.3129764]
4.84 [7.171817]
7.81 [2.5714254]
6.11 [30.600655]
7.13 [3.746866]
7.13 [3.1903343]
4.1 [10.069302]
3.56 [13.5482435]
4.1 [11.011642]
3.56 [15.846395]
5.54 [2.346916]
5.54 [61.254967]
6.01 [4.1606636]
3.75 [2.7585244]
5.7 [8.828032]
3.18 [3.6821396]
4.73 [3.7570965]
68.33 [16.7509]
81.2 [35.03124]
81.2 [5.7163744]
72.76 [20.712002]
72.76 [3.879623]
243.06 [12.172626]
243.06 [5.23203